<a href="https://colab.research.google.com/github/giorgiabrillanti/address-book/blob/main/RubricaGitHUB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json

In [2]:
class Contact:
    """Represents a single record (name, surname, phone, optional email)."""

    def __init__(self, name, surname, phone, email=""):
        self.name = name
        self.surname = surname
        self.phone = phone
        self.email = email

    def __repr__(self):
        """Returns a readable description of the contact, instead of just the object type."""
        return f"Name: {self.get_fullname()} - Phone: {self.phone} - Email: {self.get_email()}"

    def get_fullname(self):
        """Returns name and surname separated by a space."""
        return self.name + " " + self.surname

    def get_phone(self):
        """Returns the phone number."""
        return self.phone

    def get_email(self):
        """Returns the (optional) email, or "Email not available" if missing."""
        if self.email:
            return self.email
        else:
            return "Email not available"

    def to_dict(self):
        """Turns the contact into a dictionary, so it can be saved to a JSON file."""
        return {"name": self.name, "surname": self.surname,
                "phone": self.phone, "email": self.email}

In [3]:
class AddressBook:
    """Manages the collection of contacts and the operations on it."""

    def __init__(self):
        """Initializes the address book with an empty list of contacts."""
        self.contacts_list = []

    def add(self, contact):
        """Adds a contact at the end of the list."""
        self.contacts_list.append(contact)

    def view(self):
        """Returns the whole list of contacts."""
        return self.contacts_list

    def edit(self, selected, new_phone="", new_email=""):
        """Updates phone and email of the selected contact.
        Empty fields are left unchanged."""
        if new_phone:
            selected.phone = new_phone
        if new_email:
            selected.email = new_email

    def delete(self, selected):
        """Removes the selected contact from the list."""
        self.contacts_list.remove(selected)

    def search(self, query):
        """Returns the contacts whose full name contains the searched text."""
        found = []
        for contact in self.contacts_list:
            if query.lower() in contact.get_fullname().lower():
                found.append(contact)
        return found

    def save(self, path="contacts.json"):
        """Saves the contacts to a JSON file, after turning them into dictionaries."""
        data = []
        for contact in self.contacts_list:
            data.append(contact.to_dict())
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

    def load(self, path="contacts.json"):
        """Loads the contacts from the JSON file.
        If the file is missing, empty or corrupted, the address book stays empty.
        Records without name, surname or phone are skipped."""
        self.contacts_list = []  # start from empty: avoid duplicates
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except (FileNotFoundError, json.JSONDecodeError):
            return  # missing or corrupted file: the address book stays empty

        for d in data:
            try:
                contact = Contact(d["name"], d["surname"], d["phone"], d.get("email", ""))
            except (KeyError, TypeError):
                continue  # incomplete or invalid record: skip it
            self.add(contact)

In [4]:
# Helper functions

def mandatory_answer(question):
    """Asks for a non-empty value, repeating the question until the user answers.
    Returns the text entered, or None if the user types 0 to cancel."""
    while True:
        value = input(question + " (0 to cancel) ").strip()
        if value == "0":
            return None
        if value:
            return value
        print("You must enter a value in this field!")


def valid_phone(phone):
    """Returns True if the text contains only digits, + and -, and at least one digit."""
    return any(c.isdigit() for c in phone) and all(c in "0123456789+-" for c in phone)


def valid_email(email):
    """Returns True if the email has a single @, some text before it and a dot in the domain."""
    if email.count("@") != 1:
        return False
    local_part, domain = email.split("@")
    return bool(local_part) and "." in domain


def ask_valid(question, validator, optional=False):
    """Asks for a value and repeats the question until the format is valid.

    validator: function that checks the format (valid_phone, valid_email).
    optional: if True, an empty answer is allowed and "" is returned.
    Returns the value, "" if empty, or None if the user cancels with 0.
    """
    while True:
        value = input(f"{question} (0 to cancel): ").strip()
        if value == "0":
            return None
        if value == "":
            if optional:
                return ""
            print("Required field")
            continue
        if validator(value):
            return value
        print("Invalid format")


def show_numbered(contacts):
    """Prints the contacts numbered, with a counter starting from 1."""
    for i, c in enumerate(contacts, start=1):
        print(i, c)


def search_contact(address_book):
    """Asks for a text and searches the address book, repeating the question until something is found.
    Returns the list of contacts found, or None if the user cancels."""
    while True:
        query = mandatory_answer("Enter your search:")
        if query is None:
            return None
        found = address_book.search(query)
        if found:
            return found
        print("The search returned no results, try again (0 to go back to the menu)")


def choose_contact(found, action):
    """Shows the numbered contacts and lets the user pick the one to act on.
    Returns the chosen contact, or None if the user cancels."""
    show_numbered(found)

    while True:
        text = input(f"Select the number of the contact to {action} (0 to cancel): ").strip()
        if text == "0":
            return None
        try:
            number = int(text)
        except ValueError:
            print("You must enter a number")
            continue

        if 1 <= number <= len(found):
            return found[number - 1]
        print("Invalid number.")

In [5]:
def main():
    """Main loop: shows the menu and handles the user's choices."""
    address_book = AddressBook()
    address_book.load()

    while True:
        print("\n--- ADDRESS BOOK ---")
        print("1. Add contact")
        print("2. Show list")
        print("3. Search")
        print("4. Edit")
        print("5. Delete")
        print("0. Exit")

        choice = input("Choice: ").strip()

        # 1. Add a contact
        if choice == "1":
            name = mandatory_answer("Name:")
            if name is None:
                print("Operation cancelled.")
                continue

            surname = mandatory_answer("Surname:")
            if surname is None:
                print("Operation cancelled.")
                continue

            phone = ask_valid("Phone", valid_phone)
            if phone is None:
                print("Operation cancelled.")
                continue

            email = ask_valid("Email (enter = none)", valid_email, optional=True)
            if email is None:
                print("Operation cancelled.")
                continue

            address_book.add(Contact(name, surname, phone, email))
            address_book.save()
            print("Contact added successfully!")

        # 2. Full list
        elif choice == "2":
            contacts = address_book.view()
            if contacts:
                show_numbered(contacts)
            else:
                print("The address book is empty.")

        # 3. Search
        elif choice == "3":
            found = search_contact(address_book)
            if found is None:
                print("Operation cancelled.")
                continue
            show_numbered(found)

        # 4. Edit (search, choose the contact, update the non-empty fields)
        elif choice == "4":
            found = search_contact(address_book)
            if found is None:
                print("Operation cancelled.")
                continue

            selected = choose_contact(found, "edit")
            if selected is None:
                print("Operation cancelled.")
                continue

            new_phone = ask_valid("New phone (enter = keep current)", valid_phone, optional=True)
            if new_phone is None:
                print("Operation cancelled.")
                continue

            new_email = ask_valid("New email (enter = keep current)", valid_email, optional=True)
            if new_email is None:
                print("Operation cancelled.")
                continue

            if new_phone or new_email:
                address_book.edit(selected, new_phone, new_email)
                address_book.save()
                print("Contact updated successfully!")
            else:
                print("No changes applied.")

        # 5. Delete (with confirmation, default No)
        elif choice == "5":
            found = search_contact(address_book)
            if found is None:
                print("Operation cancelled.")
                continue

            selected = choose_contact(found, "delete")
            if selected is None:
                print("Operation cancelled.")
                continue

            confirm = input(f"Are you sure you want to delete this contact?\n{selected}\nConfirm? [y/N] (enter = No): ")
            if confirm.strip().lower() in ("y", "yes"):
                address_book.delete(selected)
                address_book.save()
                print("Contact deleted.")
            else:
                print("Deletion cancelled.")

        # 0. Save and exit
        elif choice == "0":
            address_book.save()
            print("Data saved. See you soon!")
            break

        else:
            print("Invalid choice, try again.")

In [6]:
### Demo

import os

contacts = [
    Contact("Misa", "Amane", "3201112233", "misa.amane@mail.it"),
    Contact("Misa", "Yagami", "3201928374"),
    Contact("Light", "Yagami", "3331234567", "light.yagami@mail.it"),
    Contact("Sayu", "Yagami", "3475556666"),
    Contact("Machi", "Komacine", "3201732894", "machi@mail.it"),
]

# Demo part 1: if the file does not exist, create an address book, add the contacts and save it

if not os.path.exists("contacts.json"):
    address_book = AddressBook()
    for contact in contacts:
        address_book.add(contact)
    address_book.save()
    print("Sample contacts created")
else:
    print("contacts.json already exists: sample contacts not created")

# Demo part 2:
# Reload the file

address_book2 = AddressBook()
address_book2.load()  # loads the default file, i.e. contacts.json
print("Contacts reloaded from file:")
show_numbered(address_book2.view())

Sample contacts created
Contacts reloaded from file:
1 Name: Misa Amane - Phone: 3201112233 - Email: misa.amane@mail.it
2 Name: Misa Yagami - Phone: 3201928374 - Email: Email not available
3 Name: Light Yagami - Phone: 3331234567 - Email: light.yagami@mail.it
4 Name: Sayu Yagami - Phone: 3475556666 - Email: Email not available
5 Name: Machi Komacine - Phone: 3201732894 - Email: machi@mail.it


In [7]:
main()


--- ADDRESS BOOK ---
1. Add contact
2. Show list
3. Search
4. Edit
5. Delete
0. Exit
Choice: 0
Data saved. See you soon!
